## Import

In [ ]:
import torch
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torch.utils.data import DataLoader

In [ ]:
import torch
import torchvision
from PIL import Image
from pathlib import Path

class Dataset(torch.utils.data.Dataset):
    def __init__(self, image_dir, label_dir):
        self.image_dir = Path(image_dir)
        self.label_dir = Path(label_dir)
        self.images = sorted(self.image_dir.glob("*.*"))

    def __getitem__(self, idx):
        img_path = self.images[idx]
        label_path = self.label_dir / (img_path.stem + ".txt")

        img = Image.open(img_path).convert("RGB")

        # Hämta bildens storlek innan to_tensor
        img_w, img_h = img.size

        img = torchvision.transforms.functional.to_tensor(img)

        boxes = []
        labels = []

        if label_path.exists():
            with open(label_path) as f:
                for line in f:
                    cls, x, y, w, h = map(float, line.split())

                    xmin = (x - w / 2) * img_w
                    ymin = (y - h / 2) * img_h
                    xmax = (x + w / 2) * img_w
                    ymax = (y + h / 2) * img_h

                    boxes.append([xmin, ymin, xmax, ymax])
                    labels.append(int(cls) + 1)

        boxes = torch.tensor(boxes, dtype=torch.float32)
        labels = torch.tensor(labels, dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels
        }

        return img, target

    def __len__(self):
        return len(self.images)


dataset = Dataset("images", "labels")
print(len(dataset))

img, target = dataset[0]

print(img.shape)
print(target)

8450


ValueError: too many values to unpack (expected 4)

In [8]:
from torch.utils.data import DataLoader

def collate_fn(batch):
    return tuple(zip(*batch))

dataset = Dataset("images", "labels")

loader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=collate_fn
)

#### Formatting (Might not be needed)

In [6]:
def Yolo_formatting(xmin, ymin, xmax, ymax, image_width, image_height):
    x_center = ((xmin + xmax) / 2) / image_width
    y_center = ((ymin + ymax) / 2) / image_height
    
    width = (xmax-xmin) / image_width
    height = (ymax - ymin) / image_height

    return x_center, y_center, width, height

## Model

In [3]:
num_classes = 2

model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights="DEFAULT")
in_features = model.roi_heads.box_predictor.cls_score.in_features

model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)